# Modelling

# Imports and Data Loading

In [407]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import statsmodels.formula.api as smf
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [408]:
train_df = pd.read_csv("../../data/processed/train.csv", keep_default_na=False)
test_df = pd.read_csv("../../data/processed/test.csv", keep_default_na=False)
val_df = pd.read_csv("../../data/processed/val.csv", keep_default_na=False)

In [409]:
train_df.shape, test_df.shape, val_df.shape

((2051, 95), (440, 95), (439, 95))

In [410]:
def compute_vif(model, train_df):
    X = model.model.exog  # design matrix
    vif_data = pd.DataFrame()
    vif_data["feature"] = model.model.exog_names
    vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    return vif_data.sort_values("VIF", ascending=False)

# Linear Regression Models

In [411]:
def build_regression_model(formula, train_df, val_df, model_number):
    
    model = smf.ols( 
        formula, 
        data=train_df
    ).fit()

    print(model.summary())

    predictions = model.predict(val_df)
    rmse = np.sqrt(mean_squared_error(val_df["Log_SalePrice"], predictions))

    predictions_original_scale = np.exp(predictions)
    rmse_original_scale = np.sqrt(mean_squared_error(np.exp(val_df["Log_SalePrice"]), predictions_original_scale))

    print("\n------ Model Performance on Validation Set ------")
    print(f"R²: {model.rsquared:.4f} | Adjusted R²: {model.rsquared_adj:.4f}")
    print(f"Model {model_number} - RMSE (Log Scale): {rmse:.4f}")
    print(f"Model {model_number} - RMSE (Original Scale): ${rmse_original_scale:,.0f}")
    
    return model

## Model 1

First model is a baseline with three numerical variables that had high Pearson correlation with Sale Price.

In [412]:
formula1 = 'Log_SalePrice ~ Q("Overall Score") + Q("House Age") + TotalSF'

model_linear1 = build_regression_model(formula1, train_df, val_df, model_number=1)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.808
Model:                            OLS   Adj. R-squared:                  0.808
Method:                 Least Squares   F-statistic:                     2872.
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:25:12   Log-Likelihood:                 646.32
No. Observations:                2051   AIC:                            -1285.
Df Residuals:                    2047   BIC:                            -1262.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.5849      0

The first model has a ${R^2}$ of 0.803, which means that 80.8% of the values can be explained by the three chosen features.

This shows that Prices are highly influenced by the quality and sizing of a property but also shows that more recent houses tend to be more expensive.

If we take a look at the House Age coefficient we can also see that it is negative which shows that the price of a housing goes down as it ages.

The std err of TotalSF is really big, in the next iteration I should probably log it so it stabilizes.

In [413]:
compute_vif(model_linear1, train_df)

,feature,VIF
0,Intercept,56.959077
3,TotalSF,1.301976
1,"Q(""Overall Score"")",1.211018
2,"Q(""House Age"")",1.141052


## Model 2

As we've seen, house age has an impact on the model's performance, In this iteration I will see if `Years Since Remodel` has a significant impact on the performance as well.

By looking at the features with more correlation with SalePrice, we can see that the Garage grouping makes a difference, so I will be testing the `Garage Score` variable as well.

In [414]:
formula2 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + np.log(TotalSF)' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \

model_linear2 = build_regression_model(formula2, train_df, val_df, model_number=2)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.834
Model:                            OLS   Adj. R-squared:                  0.833
Method:                 Least Squares   F-statistic:                     2052.
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:25:12   Log-Likelihood:                 794.14
No. Observations:                2051   AIC:                            -1576.
Df Residuals:                    2045   BIC:                            -1543.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

As we can see the second model obtained an $R^2$ of 0.8338, which means 83.4% of the values can be explained by the current features. This represent a 2.6% improvement of our model with the addition of only two extra features.

As expected, `Garage Score` coefficient indicated that it has a positive impact on the `Sale Price` value, i.e houses with better Garages are more expensive.

On the opposite direction, the `Years Since Remodel` behaves like House Age. The negative coefficient indicates that the longer it has been since a house was renovated, the cheaper the property is going to be. 

In [415]:
compute_vif(model_linear2, train_df)

,feature,VIF
0,Intercept,760.420437
4,"Q(""Years Since Remodel"")",1.933727
2,"Q(""House Age"")",1.737583
1,"Q(""Overall Score"")",1.549214
3,np.log(TotalSF),1.342678
5,"Q(""Garage Score"")",1.162022


## Model 3

In this try of the model I want to verify how the model performs if I add some of the highest correlated categorical features: `Neighbourhood` and `Condition 1` and `Condition 2` (joined in `Condition_Group`).

A good and safe Neighbourhood in a nice location (near schools, hospitals, etc), is more likely to be more expensive.

On the same thought process the `Condition` features mean a certain proximity of the property to various conditions, which can add value to the property.

In [416]:
formula3 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + np.log(TotalSF)' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood_Simplified' \
                        ' + Condition_Group' \

model_linear3 = build_regression_model(formula3, train_df, val_df, model_number=3)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.840
Model:                            OLS   Adj. R-squared:                  0.840
Method:                 Least Squares   F-statistic:                     1193.
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:25:12   Log-Likelihood:                 834.98
No. Observations:                2051   AIC:                            -1650.
Df Residuals:                    2041   BIC:                            -1594.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

The third model obtained an $R^2$ of 0.8403. Compared to the second model 83.4%, this is a 0.63% improvement of our model.

The coefficients on each categorical feature corroborate with the theory that being in different neighbourhoods or having proximity to certain conditions can make the price of the house be lower or higher.

In [417]:
compute_vif(model_linear3, train_df)

,feature,VIF
0,Intercept,778.227863
2,Neighborhood_Simplified[T.Tier_3_Luxury],3.514970
6,"Q(""House Age"")",2.757031
1,Neighborhood_Simplified[T.Tier_2_Mid],2.334666
8,"Q(""Years Since Remodel"")",1.993561
5,"Q(""Overall Score"")",1.559384
7,np.log(TotalSF),1.377907
9,"Q(""Garage Score"")",1.213202
3,Condition_Group[T.Normal],1.193074
4,Condition_Group[T.Positive],1.165997


## Model 4

In this model I will test the impact of `Has Fireplace` and `Total Bath`. 

Usually, in property listings, houses with a greater amount of bathroom mean that they are bigger and therefore more expensive.

Having a Fireplace might also be an indicator of higher priced properties as it is seen as a luxury item.

In [418]:
formula4 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + np.log(TotalSF)' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood_Simplified' \
                        ' + Condition_Group' \
                        ' + Q("Has Fireplace")' \
                        ' + Q("Total Bath")' \

model_linear4 = build_regression_model(formula4, train_df, val_df, model_number=4)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.856
Model:                            OLS   Adj. R-squared:                  0.855
Method:                 Least Squares   F-statistic:                     1104.
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:25:12   Log-Likelihood:                 942.97
No. Observations:                2051   AIC:                            -1862.
Df Residuals:                    2039   BIC:                            -1794.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

From the 3rd to the 4th model there was an improvement the value of $R^2$ (was 0.8403, now 0.8563).

Analyzing the coefficients of the newly added features, we can confirm that both of them have a positive impact on the price range i.e if a house has a fireplace, or a bigger amount of bathrooms, the price will be bigger.

In [419]:
compute_vif(model_linear4, train_df)

,feature,VIF
0,Intercept,1015.377629
2,Neighborhood_Simplified[T.Tier_3_Luxury],3.537453
6,"Q(""House Age"")",3.027162
1,Neighborhood_Simplified[T.Tier_2_Mid],2.346755
8,"Q(""Years Since Remodel"")",2.024515
11,"Q(""Total Bath"")",2.021257
7,np.log(TotalSF),1.982539
5,"Q(""Overall Score"")",1.584067
10,"Q(""Has Fireplace"")",1.334520
9,"Q(""Garage Score"")",1.228915


## Model 5

`Mas Vnr Area` and `Mas Vnr Type` also seem to have some sort of impact in the value of Sale Price accordingly to the correlation analysis, let's see if that translates to an improvement of our model.

In [420]:
formula5 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + np.log(TotalSF)' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood_Simplified' \
                        ' + Condition_Group' \
                        ' + Q("Has Fireplace")' \
                        ' + Q("Total Bath")' \
                        ' + Q("Mas Vnr Area")' \
                        ' + Q("Mas Vnr Type")' \

model_linear5 = build_regression_model(formula5, train_df, val_df, model_number=5)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.859
Model:                            OLS   Adj. R-squared:                  0.858
Method:                 Least Squares   F-statistic:                     776.7
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:25:12   Log-Likelihood:                 965.21
No. Observations:                2051   AIC:                            -1896.
Df Residuals:                    2034   BIC:                            -1801.
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

Altough the model improved by 0.1% the features don't seem to have a significant impact on the model, especially with Mas Vnr Area having such high std err

In [421]:
compute_vif(model_linear5, train_df)

,feature,VIF
0,Intercept,1164.234077
7,"Q(""Mas Vnr Type"")[T.None]",27.052046
5,"Q(""Mas Vnr Type"")[T.BrkFace]",23.986953
8,"Q(""Mas Vnr Type"")[T.Stone]",8.909035
2,Neighborhood_Simplified[T.Tier_3_Luxury],3.559179
10,"Q(""House Age"")",3.202351
1,Neighborhood_Simplified[T.Tier_2_Mid],2.395851
16,"Q(""Mas Vnr Area"")",2.144597
11,np.log(TotalSF),2.098953
12,"Q(""Years Since Remodel"")",2.060013


## Model 6

in this model I have decided to test the `MS SubClass` and `MS Zoning` feature alongside `Mas Vnr Type` (that altough didn't have explendid results was still able to improve the model slightly).

`MS SubClass` might raise or lower the property price since different dwellings might be more appealing or not.

`MS Zoning` refers to the zone of the property. A house in an industrial zone might not be as appealing as one in a Residential Area.

In [422]:
formula6 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + np.log(TotalSF)' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood_Simplified' \
                        ' + Condition_Group' \
                        ' + Q("Has Fireplace")' \
                        ' + Q("Total Bath")' \
                        ' + Q("MS SubClass")' \
                        ' + Q("MS Zoning")' \

model_linear6 = build_regression_model(formula6, train_df, val_df, model_number=6)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.862
Model:                            OLS   Adj. R-squared:                  0.861
Method:                 Least Squares   F-statistic:                     706.8
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:25:12   Log-Likelihood:                 986.76
No. Observations:                2051   AIC:                            -1936.
Df Residuals:                    2032   BIC:                            -1829.
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

The 5th model has a 0.8623 $R^2$, the greatest so far. The addition of `MS SubClass` and `MS Zoning` greatly improved the model as expected, with different zones and types of property having different impacts on the price of the house.

In [423]:
compute_vif(model_linear6, train_df)

,feature,VIF
0,Intercept,1941.885073
9,"Q(""MS Zoning"")[T.RL]",186.214994
10,"Q(""MS Zoning"")[T.RM]",145.776330
6,"Q(""MS Zoning"")[T.FV]",44.647669
8,"Q(""MS Zoning"")[T.RH]",12.653078
5,"Q(""MS Zoning"")[T.C (all)]",10.581355
2,Neighborhood_Simplified[T.Tier_3_Luxury],4.350236
1,Neighborhood_Simplified[T.Tier_2_Mid],3.372549
12,"Q(""House Age"")",3.080283
17,"Q(""Total Bath"")",2.172157


## Model 7 

For this last attempt I will see if `Sale Condition` has an impact on the model.

A Sale Condition of 'Partial' means that the house was not completed when the house was sold, which means the house is not yet fully built and therefore is very new, which can make it more expensive.

In [424]:
formula7 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + np.log(TotalSF)' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood_Simplified' \
                        ' + Condition_Group' \
                        ' + Q("Has Fireplace")' \
                        ' + Q("Total Bath")' \
                        ' + Q("MS SubClass")' \
                        ' + Q("Sale Condition")' \

model_linear7 = build_regression_model(formula7, train_df, val_df, model_number=7)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.864
Model:                            OLS   Adj. R-squared:                  0.863
Method:                 Least Squares   F-statistic:                     758.5
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:25:13   Log-Likelihood:                 998.24
No. Observations:                2051   AIC:                            -1960.
Df Residuals:                    2033   BIC:                            -1859.
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

This model proves to be the best with an $R^2$ of 0.8638 (86,4% of values are explained by the features chosen).

As expected, the `Sale Condition` feature significantly improved the model meaning that different conditions during the sale have different impacts on the Sale Price.

The RMSE in the original scale being 23,619 is an acceptable error considering how big house pricings can be.

In [425]:
compute_vif(model_linear7, train_df)

,feature,VIF
0,Intercept,1082.028490
2,Neighborhood_Simplified[T.Tier_3_Luxury],3.622309
11,"Q(""House Age"")",3.145596
1,Neighborhood_Simplified[T.Tier_2_Mid],2.519837
8,"Q(""Sale Condition"")[T.Normal]",2.407459
9,"Q(""Sale Condition"")[T.Partial]",2.334019
16,"Q(""Total Bath"")",2.188295
13,"Q(""Years Since Remodel"")",2.064295
12,np.log(TotalSF),2.064159
10,"Q(""Overall Score"")",1.600481


## Model 8

As we've seen, adding the number of Bathrooms highly improved the results, let's see if the same happens by adding `Bedroom AbvGr`

In [426]:
formula8 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + np.log(TotalSF)' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood_Simplified' \
                        ' + Condition_Group' \
                        ' + Q("Has Fireplace")' \
                        ' + Q("Total Bath")' \
                        ' + Q("MS SubClass")' \
                        ' + Q("Sale Condition")' \
                        ' + Q("Bedroom AbvGr")' \

model_linear8 = build_regression_model(formula8, train_df, val_df, model_number=8)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.864
Model:                            OLS   Adj. R-squared:                  0.863
Method:                 Least Squares   F-statistic:                     716.0
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:25:13   Log-Likelihood:                 998.24
No. Observations:                2051   AIC:                            -1958.
Df Residuals:                    2032   BIC:                            -1852.
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

Altough the $R^2$ improved the tiniest bit, the Ajusted $R^2$ got lower which means it is probably not worth it to add any more features to this model.

# Logistic Regression

In [427]:
def build_logistic_regression_model(formula, train_df, val_df, model_number):
    """
    Builds a logistic regression model, evaluates it using: 
    - F1 score
    - Accuracy
    - ROC-AUC
    and prints the results.
    """

    model = smf.logit( 
        formula, 
        data=train_df
    ).fit()

    print(model.summary())

    predictions = model.predict(val_df)
    predicted_classes = (predictions >= 0.5).astype(int)

    f1 = f1_score(val_df["Is High Price"], predicted_classes)
    accuracy = accuracy_score(val_df["Is High Price"], predicted_classes)
    roc_auc = roc_auc_score(val_df["Is High Price"], predictions)


    print("-------- Model Evaluation: --------")
    print(f"Model {model_number} - Accuracy: {accuracy:.4f}")

    print(f"Model {model_number} - F1 Score: {f1:.4f}")

    print(f"Model {model_number} - ROC-AUC: {roc_auc:.4f}")

    return model

## Model 1

For this first attemp I want to try to use the same baseline features () I used for the Linear Regression since they had an amazing result as a first try.

In [428]:
formula1_lr = 'Q("Is High Price") ~ Q("Overall Score") + Q("House Age") + np.log(TotalSF)'

model_logit1 = build_logistic_regression_model(formula1_lr, train_df, val_df, model_number=1)

Optimization terminated successfully.
         Current function value: 0.232303
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:     Q("Is High Price")   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2047
Method:                           MLE   Df Model:                            3
Date:                Sun, 19 Apr 2026   Pseudo R-squ.:                  0.6347
Time:                        17:25:13   Log-Likelihood:                -476.45
converged:                       True   LL-Null:                       -1304.3
Covariance Type:            nonrobust   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            -86.8617      4.733    -18.352      0.000     -96.138     -77.585
Q("Ov

Like in the Linear Regression model, these three features are really good at predicting the price of the property.

- An Accuracy of 0.9408 means that ~94% of the time the model guessed the price tier correctly.
- An F1 Score of 0.9182 means that the model is robust against missing high-priced houses and avoiding false alarms.
- A ROC-AUC of 0.9887 is amazing for a first try as it means that if you randomly picked one high-price house and one low-price house, the model would correctly rank the high-price one higher ~98.87% of the time. 

In [429]:
compute_vif(model_logit1, train_df)

,feature,VIF
0,Intercept,736.750557
3,np.log(TotalSF),1.337927
1,"Q(""Overall Score"")",1.227207
2,"Q(""House Age"")",1.154245


## Model 2

Let's try to use some binary features like `Has Garage` and `Has Fireplace` to try to see if the model improves.

In [430]:
formula2_lr = 'Q("Is High Price") ~ Q("Overall Score")' \
                                ' + Q("House Age") ' \
                                ' + np.log(TotalSF) ' \
                                ' + Q("Has Garage")' \
                                ' + Q("Has Fireplace")' \

model_logit2 = build_logistic_regression_model(formula2_lr, train_df, val_df, model_number=2)

Optimization terminated successfully.
         Current function value: 0.226020
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:     Q("Is High Price")   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2045
Method:                           MLE   Df Model:                            5
Date:                Sun, 19 Apr 2026   Pseudo R-squ.:                  0.6446
Time:                        17:25:13   Log-Likelihood:                -463.57
converged:                       True   LL-Null:                       -1304.3
Covariance Type:            nonrobust   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            -86.5769      5.082    -17.037      0.000     -96.537     -76.617
Q("Ov

Looking at the overall results we can see that there was a decrease in overall performance let's try to think why:

Has Garage  having a $p=0.117$ means that this feature is "statistically insignificant" at the standard 95% confidence level, althought on paper having a coef = 1.6651 has a huge impact, but if we look at the Standard Error (1.064). It’s so high that the model isn't confident in that 1.66 coefficient. This is why the p-value is poor.

On the contrary Has Fireplace has a $p=0.000$. This is highly significant and it suggests that it helps the model.





In [431]:
compute_vif(model_logit2, train_df)

,feature,VIF
0,Intercept,844.771906
3,np.log(TotalSF),1.510478
5,"Q(""Has Fireplace"")",1.297442
1,"Q(""Overall Score"")",1.272648
2,"Q(""House Age"")",1.185376
4,"Q(""Has Garage"")",1.102690


## Model 3

Let's remove the `Has Garage` feature and add features relating to the number of bathrooms (`Total Bath`) and `Years Since Remodel`.

In [432]:
formula3_lr = 'Q("Is High Price") ~ Q("Overall Score")' \
                                ' + Q("House Age") ' \
                                ' + np.log(TotalSF) ' \
                                ' + Q("Has Fireplace")' \
                                ' + Q("Total Bath")' \
                                ' + Q("Years Since Remodel")' \

model_logit3 = build_logistic_regression_model(formula3_lr, train_df, val_df, model_number=3)

Optimization terminated successfully.
         Current function value: 0.218846
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:     Q("Is High Price")   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2044
Method:                           MLE   Df Model:                            6
Date:                Sun, 19 Apr 2026   Pseudo R-squ.:                  0.6559
Time:                        17:25:13   Log-Likelihood:                -448.85
converged:                       True   LL-Null:                       -1304.3
Covariance Type:            nonrobust   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                  -81.2871      4.927    -16.497      0.000     -90.945

This is the best model so far.

`Total Bath` has a Coef:  0.6494, this is a strong positive predictor. It suggests that the number of bathrooms is a major "tier-breaker" for house prices.
Like in the Linear Regression, `Years Since Remodel` has a positive impact in the performance of the model, confirming the idea that a house being "refreshed" is a big indicator for higher prices. 

In [433]:
compute_vif(model_logit3, train_df)

,feature,VIF
0,Intercept,995.888450
5,"Q(""Total Bath"")",1.992313
6,"Q(""Years Since Remodel"")",1.947284
3,np.log(TotalSF),1.930354
2,"Q(""House Age"")",1.818764
1,"Q(""Overall Score"")",1.515373
4,"Q(""Has Fireplace"")",1.292850


## Model 4


In [434]:
formula4_lr = 'Q("Is High Price") ~ Q("Overall Score")' \
                                ' + Q("House Age") ' \
                                ' + np.log(TotalSF) ' \
                                ' + Q("Has Fireplace")' \
                                ' + Q("Total Bath")' \
                                ' + Q("Years Since Remodel")' \
                                ' + Q("Neighborhood Rank")' \

model_logit4 = build_logistic_regression_model(formula4_lr, train_df, val_df, model_number=4)

Optimization terminated successfully.
         Current function value: 0.204681
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:     Q("Is High Price")   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2043
Method:                           MLE   Df Model:                            7
Date:                Sun, 19 Apr 2026   Pseudo R-squ.:                  0.6781
Time:                        17:25:13   Log-Likelihood:                -419.80
converged:                       True   LL-Null:                       -1304.3
Covariance Type:            nonrobust   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                  -78.8065      5.074    -15.532      0.000     -88.751

In the Linear Regression the Neighbourhood was a good predictor of price, here it appears to have same effect.

It is to take in consideration that ROC-AUC value went down with the addition of one more feature, which can mean it is overfitting.

In [435]:
compute_vif(model_logit4, train_df)

,feature,VIF
0,Intercept,1041.755484
2,"Q(""House Age"")",2.694385
7,"Q(""Neighborhood Rank"")",2.653695
3,np.log(TotalSF),2.068754
5,"Q(""Total Bath"")",1.992346
6,"Q(""Years Since Remodel"")",1.949559
1,"Q(""Overall Score"")",1.564365
4,"Q(""Has Fireplace"")",1.304812


## Model 5


In [436]:
formula5_lr = 'Q("Is High Price") ~ Q("Overall Score")' \
                                ' + Q("House Age") ' \
                                ' + np.log(TotalSF) ' \
                                ' + Q("Has Fireplace")' \
                                ' + Q("Total Bath")' \
                                ' + Q("Years Since Remodel")' \
                                ' + Q("Neighborhood Rank")' \
                                ' + Q("Bedroom AbvGr")' \

model_logit5 = build_logistic_regression_model(formula5_lr, train_df, val_df, model_number=5)

Optimization terminated successfully.
         Current function value: 0.204651
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:     Q("Is High Price")   No. Observations:                 2051
Model:                          Logit   Df Residuals:                     2042
Method:                           MLE   Df Model:                            8
Date:                Sun, 19 Apr 2026   Pseudo R-squ.:                  0.6782
Time:                        17:25:13   Log-Likelihood:                -419.74
converged:                       True   LL-Null:                       -1304.3
Covariance Type:            nonrobust   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                  -78.4684      5.159    -15.211      0.000     -88.579

By adding even more features the results seem to go lower, I will assume the model is overfitting and take Model 4 as the best one.

In [437]:
compute_vif(model_logit5, train_df)

,feature,VIF
0,Intercept,1179.727754
2,"Q(""House Age"")",2.725872
7,"Q(""Neighborhood Rank"")",2.698417
3,np.log(TotalSF),2.495232
5,"Q(""Total Bath"")",2.017179
6,"Q(""Years Since Remodel"")",1.951223
1,"Q(""Overall Score"")",1.571314
8,"Q(""Bedroom AbvGr"")",1.354002
4,"Q(""Has Fireplace"")",1.318078


# Testing Models

In [438]:
test_df['pred_linear'] = model_linear7.predict(test_df)
test_rmse = np.sqrt(mean_squared_error(test_df["Log_SalePrice"], test_df["pred_linear"]))

test_df['pred_linear_original'] = np.exp(test_df['pred_linear'])
test_rmse_original = np.sqrt(mean_squared_error(np.exp(test_df["Log_SalePrice"]), test_df["pred_linear_original"]))

print(f"Test RMSE (Log Scale): {test_rmse:.4f}")
print(f"Test RMSE (Original Scale): ${test_rmse_original:,.0f}")

Test RMSE (Log Scale): 0.1436
Test RMSE (Original Scale): $36,586


In [439]:
test_df['pred_logit'] = model_logit4.predict(test_df)

predicted_classes = (test_df['pred_logit'] >= 0.5).astype(int)

f1 = f1_score(test_df["Is High Price"], predicted_classes)
accuracy = accuracy_score(test_df["Is High Price"], predicted_classes)
roc_auc = roc_auc_score(test_df["Is High Price"], test_df['pred_logit'])


print("-------- Model Evaluation: --------")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

-------- Model Evaluation: --------
Accuracy: 0.9159
F1 Score: 0.8869
ROC-AUC: 0.9757
